# Assignment 2 — Reward Model (TRL + LoRA + HH-RLHF)

This notebook trains a **Reward Model (RM)** using **Hugging Face TRL** and **PEFT (LoRA)** on the **Anthropic HH-RLHF** preference dataset.

This version is made for **Google Colab**. It mounts **Google Drive** and saves the model and result files there.

What you get:
- A reproducible reward-model training run
- A small reward model built from a causal LM backbone plus a scalar reward head
- Saved model/adapters to disk
- Pairwise reward scores on fixed evaluation pairs, saved to `results/rm_pair_scores.json`

Docs:
- TRL RewardTrainer docs: https://huggingface.co/docs/trl/en/reward_trainer
- HH-RLHF dataset: https://huggingface.co/datasets/Anthropic/hh-rlhf

> 1.5B reward-model backbone: `Qwen/Qwen2.5-1.5B`.


## Step 1: Install packages

Run this cell in Colab first. If Colab asks, restart the runtime after install.


In [ ]:
# Install packages for Colab.
# If Colab says restart runtime, do that and then run the notebook again.
!pip -q install -U "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
                  "trl==0.29.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" "sentencepiece" "wandb" "evaluate"

## Step 2: Mount Google Drive

This step connects Colab to your Google Drive. The trained model and result files will be saved there.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive is ready.')
except ImportError:
    print('This notebook is not running in Google Colab.')


## Step 3: Import and set basic options

This part imports the tools, sets the random seed, chooses the model and dataset, and creates folders in Google Drive.


In [ ]:
import gc
import inspect
import os, json, random
from datetime import datetime

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed
from peft import LoraConfig, PeftModel
from trl import RewardConfig, RewardTrainer

# -----------------------
# Reproducibility
# -----------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# -----------------------
# Model choice
# -----------------------
MODEL_NAME = "Qwen/Qwen2.5-1.5B"
# MODEL_NAME = "Qwen/Qwen2.5-0.5B"  # smaller alternative if you need less VRAM

# -----------------------
# Dataset choice
# -----------------------
RM_DATASET = "Anthropic/hh-rlhf"

# Keep this notebook practical for Colab and target about 40 minutes on an A100.
# FAST_MODE is a lighter fallback. The default below is tuned for a better speed/accuracy tradeoff.
FAST_MODE = False
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
HIGH_VRAM_GPU = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory >= 30 * 1024**3
NUM_TRAIN_EPOCHS = 1

if FAST_MODE:
    MAX_TRAIN_SAMPLES = 3072
    MAX_EVAL_SAMPLES = 512
    MAX_LENGTH = 384
    PER_DEVICE_TRAIN_BATCH_SIZE = 2 if HIGH_VRAM_GPU else 1
    PER_DEVICE_EVAL_BATCH_SIZE = 4 if HIGH_VRAM_GPU else 1
    GRADIENT_ACCUMULATION_STEPS = 8 if HIGH_VRAM_GPU else 16
    LEARNING_RATE = 2e-5
    WARMUP_STEPS = 30
    LOGGING_STEPS = 10
    EVAL_STRATEGY = "steps"
    EVAL_STEPS = 150
    SAVE_STRATEGY = "steps"
    SAVE_STEPS = 150
    SAVE_TOTAL_LIMIT = 2
else:
    MAX_TRAIN_SAMPLES = 9216
    MAX_EVAL_SAMPLES = 512
    MAX_LENGTH = 448
    PER_DEVICE_TRAIN_BATCH_SIZE = 2 if HIGH_VRAM_GPU else 1
    PER_DEVICE_EVAL_BATCH_SIZE = 4 if HIGH_VRAM_GPU else 1
    GRADIENT_ACCUMULATION_STEPS = 8 if HIGH_VRAM_GPU else 16
    LEARNING_RATE = 1.5e-5
    WARMUP_STEPS = 40
    LOGGING_STEPS = 25
    EVAL_STRATEGY = "steps"
    EVAL_STEPS = 256
    SAVE_STRATEGY = "steps"
    SAVE_STEPS = 256
    SAVE_TOTAL_LIMIT = 2

LOAD_BEST_MODEL_AT_END = True
ENABLE_GRADIENT_CHECKPOINTING = not HIGH_VRAM_GPU
USE_FLASH_ATTENTION = torch.cuda.is_available()

SAMPLE_PAIRS_TO_SAVE = 20
EVAL_PAIRS_TO_SCORE = min(500, MAX_EVAL_SAMPLES)
CHECKPOINT_EVAL_PAIRS = EVAL_PAIRS_TO_SCORE

# -----------------------
# Paths
# -----------------------
IN_COLAB = "COLAB_GPU" in os.environ
RUN_NAME = "reward_model_hh_rlhf_qwen25_1p5b_lora"
DRIVE_ROOT = "/content/drive/MyDrive/RLHF4LLMs"

if IN_COLAB and not os.path.exists("/content/drive/MyDrive"):
    raise ValueError("Please run the Google Drive mount cell first.")

BASE_DIR = os.path.join(DRIVE_ROOT, RUN_NAME) if IN_COLAB else os.path.abspath(RUN_NAME)
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
FINAL_MODEL_DIR = os.path.join(BASE_DIR, "final_model")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

for path in [BASE_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR, RESULTS_DIR]:
    os.makedirs(path, exist_ok=True)

print("Base folder:", BASE_DIR)
print("Checkpoints:", CHECKPOINT_DIR)
print("Final model:", FINAL_MODEL_DIR)
print("Results:", RESULTS_DIR)
print("Model:", MODEL_NAME)
print("Dataset:", RM_DATASET)
print("FAST_MODE:", FAST_MODE)
print("GPU:", GPU_NAME)
print("HIGH_VRAM_GPU:", HIGH_VRAM_GPU)

## Step 4: Load and prepare HH-RLHF

HH-RLHF gives preference pairs with `chosen` and `rejected` responses. We clean the text, drop bad rows, keep a smaller sample for Colab, and save dataset info for the report.

> Note: this dataset contains harmful, unsafe, and offensive content by design.


In [ ]:
raw_train = load_dataset(RM_DATASET, split="train")
raw_eval = load_dataset(RM_DATASET, split="test")

def normalize_pair(example):
    return {
        "chosen": example["chosen"].strip(),
        "rejected": example["rejected"].strip(),
    }

train_ds = raw_train.map(normalize_pair, remove_columns=raw_train.column_names)
eval_ds = raw_eval.map(normalize_pair, remove_columns=raw_eval.column_names)

train_ds = train_ds.filter(lambda x: x["chosen"] != "" and x["rejected"] != "" and x["chosen"] != x["rejected"])
eval_ds = eval_ds.filter(lambda x: x["chosen"] != "" and x["rejected"] != "" and x["chosen"] != x["rejected"])

if MAX_TRAIN_SAMPLES is not None:
    train_ds = train_ds.shuffle(seed=SEED).select(range(min(MAX_TRAIN_SAMPLES, len(train_ds))))
if MAX_EVAL_SAMPLES is not None:
    eval_ds = eval_ds.shuffle(seed=SEED).select(range(min(MAX_EVAL_SAMPLES, len(eval_ds))))

dataset_info = {
    "dataset_name": RM_DATASET,
    "train_rows": len(train_ds),
    "eval_rows": len(eval_ds),
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "max_eval_samples": MAX_EVAL_SAMPLES,
    "example": train_ds[0],
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_length": MAX_LENGTH,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "saved_at": datetime.now().isoformat(timespec="seconds"),
}

length_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if length_tokenizer.pad_token is None:
    length_tokenizer.pad_token = length_tokenizer.eos_token

def token_length_stats(ds, name, tokenizer, max_length):
    chosen_lengths = [len(tokenizer(example["chosen"], add_special_tokens=True, truncation=False)["input_ids"]) for example in ds]
    rejected_lengths = [len(tokenizer(example["rejected"], add_special_tokens=True, truncation=False)["input_ids"]) for example in ds]
    combined_lengths = chosen_lengths + rejected_lengths
    over_limit = sum(length > max_length for length in combined_lengths)
    return {
        "name": name,
        "num_pairs": len(ds),
        "max_length": max_length,
        "mean_tokens": round(float(np.mean(combined_lengths)), 2),
        "p95_tokens": int(np.percentile(combined_lengths, 95)),
        "max_tokens": int(np.max(combined_lengths)),
        "truncation_rate": round(over_limit / max(1, len(combined_lengths)), 4),
    }

train_length_stats = token_length_stats(train_ds.select(range(min(256, len(train_ds)))), "train_sample", length_tokenizer, MAX_LENGTH) if len(train_ds) else {}
eval_length_stats = token_length_stats(eval_ds.select(range(min(256, len(eval_ds)))), "eval_sample", length_tokenizer, MAX_LENGTH) if len(eval_ds) else {}
dataset_info["train_length_stats"] = train_length_stats
dataset_info["eval_length_stats"] = eval_length_stats

with open(os.path.join(RESULTS_DIR, "dataset_info.json"), "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print("Train size:", len(train_ds))
print("Eval size:", len(eval_ds))
print("Train length stats:", train_length_stats)
print("Eval length stats:", eval_length_stats)
train_ds[0]

## Step 5: Load tokenizer and reward model backbone

A reward model is a sequence-classification model that outputs one scalar reward for a full response sequence.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if torch.cuda.is_available() else torch.float32)
model_kwargs = {
    "num_labels": 1,
    "torch_dtype": MODEL_DTYPE,
    "device_map": "auto",
}
if USE_FLASH_ATTENTION:
    model_kwargs["attn_implementation"] = "flash_attention_2"

try:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, **model_kwargs)
except Exception as exc:
    if model_kwargs.get("attn_implementation") == "flash_attention_2":
        print("flash_attention_2 unavailable, falling back to default attention:", exc)
        model_kwargs.pop("attn_implementation", None)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, **model_kwargs)
    else:
        raise

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
if ENABLE_GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
else:
    model.gradient_checkpointing_disable()

print(type(model).__name__)
print("Pad token id:", model.config.pad_token_id)
print("BF16:", USE_BF16)
print("Gradient checkpointing:", ENABLE_GRADIENT_CHECKPOINTING)
print("Flash attention requested:", USE_FLASH_ATTENTION)

## Step 6: Add LoRA

We fine-tune a small adapter instead of all model weights. The `score` layer is saved because it is the new scalar reward head.


In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["score"],
)

## Step 7: Set training options

This part builds the reward trainer. The settings below are conservative so the notebook is more likely to fit on a small Colab GPU.


In [ ]:
reward_config_kwargs = {
    "output_dir": CHECKPOINT_DIR,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "warmup_steps": WARMUP_STEPS,
    "logging_steps": LOGGING_STEPS,
    "eval_strategy": EVAL_STRATEGY,
    "eval_steps": EVAL_STEPS,
    "save_strategy": SAVE_STRATEGY,
    "save_steps": SAVE_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "load_best_model_at_end": LOAD_BEST_MODEL_AT_END,
    "max_length": MAX_LENGTH,
    "bf16": USE_BF16,
    "fp16": torch.cuda.is_available() and not USE_BF16,
    "group_by_length": True,
    "dataloader_num_workers": 2,
    "dataloader_pin_memory": torch.cuda.is_available(),
    "eval_accumulation_steps": 8,
    "save_only_model": True,
    "report_to": ["tensorboard"],
    "run_name": RUN_NAME,
    "center_rewards_coefficient": 0.01,
}

supported_reward_config_keys = set(inspect.signature(RewardConfig.__init__).parameters)
filtered_reward_config_kwargs = {
    key: value for key, value in reward_config_kwargs.items() if key in supported_reward_config_keys
}
ignored_reward_config_keys = sorted(set(reward_config_kwargs) - set(filtered_reward_config_kwargs))
if ignored_reward_config_keys:
    print("RewardConfig ignores unsupported args in this environment:", ignored_reward_config_keys)

reward_args = RewardConfig(**filtered_reward_config_kwargs)

trainer = RewardTrainer(
    model=model,
    args=reward_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

## Step 8: Train the reward model

If training stops, this cell can resume from the latest checkpoint.


In [ ]:
print("MAX_TRAIN_SAMPLES =", MAX_TRAIN_SAMPLES)
print("MAX_EVAL_SAMPLES =", MAX_EVAL_SAMPLES)
print("len(train_ds) =", len(train_ds))
print("len(eval_ds) =", len(eval_ds))
print("gradient_accumulation_steps =", GRADIENT_ACCUMULATION_STEPS)
print("per_device_train_batch_size =", reward_args.per_device_train_batch_size)
print("per_device_eval_batch_size =", reward_args.per_device_eval_batch_size)
print("num_train_epochs =", reward_args.num_train_epochs)
print("learning_rate =", reward_args.learning_rate)
print("max_length =", reward_args.max_length)
print("save_total_limit =", reward_args.save_total_limit)
print("load_best_model_at_end =", reward_args.load_best_model_at_end)
print("gradient_checkpointing =", ENABLE_GRADIENT_CHECKPOINTING)


In [ ]:
MANUAL_RESUME_CHECKPOINT = None

def find_latest_checkpoint(checkpoint_dir):
    if not os.path.isdir(checkpoint_dir):
        return None
    checkpoint_names = [name for name in os.listdir(checkpoint_dir) if name.startswith("checkpoint-")]
    if not checkpoint_names:
        return None
    checkpoint_names = sorted(checkpoint_names, key=lambda x: int(x.split("-")[-1]))
    return os.path.join(checkpoint_dir, checkpoint_names[-1])

resume_checkpoint = None
if MANUAL_RESUME_CHECKPOINT and os.path.isdir(MANUAL_RESUME_CHECKPOINT):
    resume_checkpoint = MANUAL_RESUME_CHECKPOINT
else:
    resume_checkpoint = find_latest_checkpoint(CHECKPOINT_DIR)

if resume_checkpoint:
    print("Resuming training from:", resume_checkpoint)
    train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
else:
    print("Starting training from scratch.")
    train_result = trainer.train()

train_metrics = dict(train_result.metrics)
train_metrics["model_name"] = MODEL_NAME
train_metrics["dataset_name"] = RM_DATASET
train_metrics["resume_from_checkpoint"] = resume_checkpoint

metrics_path = os.path.join(RESULTS_DIR, "train_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(train_metrics, f, ensure_ascii=False, indent=2)

train_metrics

## Step 9: Evaluate pairwise preference accuracy

We score both `chosen` and `rejected` texts, save the full scored evaluation subset to disk, and measure how often the chosen text gets the higher reward.


In [ ]:
def score_text(model, tokenizer, text, max_length=MAX_LENGTH):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    )
    model_device = next(model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    return float(logits.squeeze().float().cpu().item())

def evaluate_pairwise_accuracy(model, tokenizer, dataset, max_pairs, max_length=MAX_LENGTH, save_scores_path=None):
    eval_subset = dataset.select(range(min(max_pairs, len(dataset))))
    correct = 0
    scored_pairs = []

    model.eval()
    for example in eval_subset:
        chosen_reward = score_text(model, tokenizer, example["chosen"], max_length=max_length)
        rejected_reward = score_text(model, tokenizer, example["rejected"], max_length=max_length)
        preferred_correct = chosen_reward > rejected_reward
        correct += int(preferred_correct)
        scored_pairs.append({
            "pair_id": len(scored_pairs),
            "chosen": example["chosen"],
            "rejected": example["rejected"],
            "chosen_reward": chosen_reward,
            "rejected_reward": rejected_reward,
            "preferred_correct": preferred_correct,
        })

    pairwise_accuracy = correct / max(1, len(scored_pairs))
    if save_scores_path is not None:
        with open(save_scores_path, "w", encoding="utf-8") as f:
            json.dump(scored_pairs, f, ensure_ascii=False, indent=2)
    return pairwise_accuracy, scored_pairs

def find_all_checkpoints(checkpoint_dir):
    if not os.path.isdir(checkpoint_dir):
        return []
    checkpoint_names = [name for name in os.listdir(checkpoint_dir) if name.startswith("checkpoint-")]
    checkpoint_names = sorted(checkpoint_names, key=lambda x: int(x.split("-")[-1]))
    return [os.path.join(checkpoint_dir, name) for name in checkpoint_names]

def load_checkpoint_model(checkpoint_path):
    checkpoint_model_kwargs = {
        "num_labels": 1,
        "torch_dtype": MODEL_DTYPE,
        "device_map": "auto",
    }
    if USE_FLASH_ATTENTION:
        checkpoint_model_kwargs["attn_implementation"] = "flash_attention_2"
    try:
        base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, **checkpoint_model_kwargs)
    except Exception:
        checkpoint_model_kwargs.pop("attn_implementation", None)
        base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, **checkpoint_model_kwargs)
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.use_cache = False
    loaded_model = PeftModel.from_pretrained(base_model, checkpoint_path, is_trainable=False)
    loaded_model.eval()
    return loaded_model

checkpoint_paths = find_all_checkpoints(CHECKPOINT_DIR)
checkpoint_scores = []
best_checkpoint_path = None
best_checkpoint_accuracy = float("-inf")
best_reward_model = trainer.model

if checkpoint_paths:
    print(f"Evaluating {len(checkpoint_paths)} saved checkpoint(s) on {CHECKPOINT_EVAL_PAIRS} eval pair(s)...")
    for checkpoint_path in checkpoint_paths:
        checkpoint_model = load_checkpoint_model(checkpoint_path)
        checkpoint_accuracy, _ = evaluate_pairwise_accuracy(
            checkpoint_model,
            tokenizer,
            eval_ds,
            CHECKPOINT_EVAL_PAIRS,
            max_length=MAX_LENGTH,
        )
        checkpoint_scores.append({
            "checkpoint_path": checkpoint_path,
            "pairwise_accuracy": checkpoint_accuracy,
        })
        print(os.path.basename(checkpoint_path), "pairwise_accuracy =", round(checkpoint_accuracy, 4))
        if checkpoint_accuracy > best_checkpoint_accuracy:
            best_checkpoint_accuracy = checkpoint_accuracy
            best_checkpoint_path = checkpoint_path
        del checkpoint_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    trainer.model = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    best_reward_model = load_checkpoint_model(best_checkpoint_path)
else:
    print("No saved checkpoints found. Falling back to trainer.model for evaluation.")

eval_pair_scores_path = os.path.join(RESULTS_DIR, "eval_pair_scores.json")
pairwise_accuracy, scored_pairs = evaluate_pairwise_accuracy(
    best_reward_model,
    tokenizer,
    eval_ds,
    EVAL_PAIRS_TO_SCORE,
    max_length=MAX_LENGTH,
    save_scores_path=eval_pair_scores_path,
)

eval_metrics = {
    "pairwise_accuracy": pairwise_accuracy,
    "num_pairs": len(scored_pairs),
    "eval_pairs_to_score": EVAL_PAIRS_TO_SCORE,
    "checkpoint_eval_pairs": CHECKPOINT_EVAL_PAIRS,
    "best_checkpoint_path": best_checkpoint_path,
    "best_checkpoint_pairwise_accuracy": best_checkpoint_accuracy if best_checkpoint_path else pairwise_accuracy,
    "evaluated_at": datetime.now().isoformat(timespec="seconds"),
}

with open(os.path.join(RESULTS_DIR, "checkpoint_scores.json"), "w", encoding="utf-8") as f:
    json.dump(checkpoint_scores, f, ensure_ascii=False, indent=2)

with open(os.path.join(RESULTS_DIR, "eval_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(eval_metrics, f, ensure_ascii=False, indent=2)

print("Saved full eval pair scores to:", eval_pair_scores_path)
eval_metrics

## Step 10: Save the trained reward model

This saves the final model, tokenizer, and basic run metadata.


In [ ]:
best_reward_model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

save_info = {
    "base_dir": BASE_DIR,
    "checkpoint_dir": CHECKPOINT_DIR,
    "final_model_dir": FINAL_MODEL_DIR,
    "results_dir": RESULTS_DIR,
    "best_checkpoint_path": best_checkpoint_path,
    "best_checkpoint_pairwise_accuracy": best_checkpoint_accuracy if best_checkpoint_path else pairwise_accuracy,
    "saved_at": datetime.now().isoformat(timespec="seconds"),
}

with open(os.path.join(RESULTS_DIR, "save_info.json"), "w", encoding="utf-8") as f:
    json.dump(save_info, f, ensure_ascii=False, indent=2)

print("Saved final model to:", FINAL_MODEL_DIR)
print("Saved result files to:", RESULTS_DIR)

## Step 11: Save fixed pair scores

We save a few evaluation pairs with both rewards so you can inspect whether the model usually prefers the better answer.


In [ ]:
sample_pairs = eval_ds.select(range(min(SAMPLE_PAIRS_TO_SAVE, len(eval_ds))))
rm_pair_scores = []

for i, example in enumerate(sample_pairs):
    chosen_reward = score_text(best_reward_model, tokenizer, example["chosen"])
    rejected_reward = score_text(best_reward_model, tokenizer, example["rejected"])
    rm_pair_scores.append({
        "pair_id": i,
        "chosen": example["chosen"],
        "rejected": example["rejected"],
        "chosen_reward": chosen_reward,
        "rejected_reward": rejected_reward,
        "preferred_correct": chosen_reward > rejected_reward,
    })

pair_scores_path = os.path.join(RESULTS_DIR, "rm_pair_scores.json")
with open(pair_scores_path, "w", encoding="utf-8") as f:
    json.dump(rm_pair_scores, f, ensure_ascii=False, indent=2)

print("Pair scores saved to:", pair_scores_path)
rm_pair_scores[0]

## Optional TensorBoard view

You can run this in Colab to watch training logs.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir {CHECKPOINT_DIR}

In [ ]:
# import os
# import matplotlib.pyplot as plt
# from tensorboard.backend.event_processing import event_accumulator

# # TensorBoard日志目录
# log_dir = CHECKPOINT_DIR + "/runs/Mar12_04-30-38_2ee1f73ef187"   # 根据你的路径修改

# # 输出目录
# save_dir = BASE_DIR + "/figures"
# os.makedirs(save_dir, exist_ok=True)

# # 读取event文件
# ea = event_accumulator.EventAccumulator(log_dir)
# ea.Reload()

# # 获取所有scalar标签
# tags = ea.Tags()["scalars"]

# print("Found scalars:", tags)

# for tag in tags:
#     events = ea.Scalars(tag)

#     steps = [e.step for e in events]
#     values = [e.value for e in events]

#     plt.figure()
#     plt.plot(steps, values)
#     plt.xlabel("Training Step")
#     plt.ylabel(tag)
#     plt.title(tag)
#     plt.grid(True)

#     filename = tag.replace("/", "_") + ".png"
#     plt.savefig(os.path.join(save_dir, filename), dpi=300)
#     plt.close()

# print("All curves saved to:", save_dir)